In [13]:
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.common.by import By

# 1. Configuracion de metadatos y conexion
NOMBRE_GRUPO = "G2_RealEstate"
CATEGORIA_ACTUAL = "Travel" # Los alumnos pueden cambiar esto segun su asignacion
try:
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['datos_libros']
    print("CONEXION EXITOSA: Listo para insertar datos de", CATEGORIA_ACTUAL)
except Exception as e:
    print("ERROR DE CONEXION:", e)

# 2. Ejemplo de logica de captura (Simplificada para la clase)
# Supongamos que ’lista_libros’ es el resultado de tu busqueda con Selenium
lista_libros = [
    {"titulo": "It’s Only Himalaya", "precio": " 45 .17"},
    {"titulo": "Full Moon over N o a h s Ark", "precio": " 49 .43"}
]

for libro in lista_libros:
    try:
    # Enriquecimiento del diccionario
        registro = {
            "item": libro["titulo"],
            "valor": float(libro["precio"].replace(' ', '').replace(',', '').strip()),
            "categoria": CATEGORIA_ACTUAL,
            "grupo": NOMBRE_GRUPO,
            "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
        }

        coleccion.insert_one(registro)
    except Exception as e:
        print("Error procesando libro:", e)

print("PROCESO FINALIZADO: Datos enviados a MongoDB.")

CONEXION EXITOSA: Listo para insertar datos de Travel
PROCESO FINALIZADO: Datos enviados a MongoDB.


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Cargar datos
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

# Esto nos dice que columnas entendio Spark que habia en Mongo
df.printSchema()

# Esto nos muestra las primeras 5 filas para confirmar que hay datos reales
df.show(5)

from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+------------+--------------------+-----+
|                 _id|categoria|      fecha_captura|       grupo|                item|valor|
+--------------------+---------+-------------------+------------+--------------------+-----+
|69d657fbf3c93f868...|   Travel|2026-04-08 13:28:27|G1_Ecommerce|  It’s Only Himalaya|45.17|
|69d657fbf3c93f868...|   Travel|2026-04-08 13:28:27|G1_Ecommerce|Full Moon over N ...|49.43|
+--------------------+---------+-------------------+------------+--------------------+-----+

PASO 2: Limpieza completada.
Registros originales: 2
Registros validos: 2


In [14]:
df.select("item", "valor").show()
df.filter(df["valor"] > 100).show()
df.groupBy("grupo").count().show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
+--------------------+-----+

+---+---------+-------------+-----+----+-----+
|_id|categoria|fecha_captura|grupo|item|valor|
+---+---------+-------------+-----+----+-----+
+---+---------+-------------+-----+----+-----+

+-------------+-----+
|        grupo|count|
+-------------+-----+
|G2_RealEstate|    2|
| G1_Ecommerce|    2|
+-------------+-----+



In [7]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+---------------+-------------+-------------+
|categoria|cantidad_libros|precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+---------------+-------------+-------------+
|   Travel|              2|           47.3|        45.17|        49.43|
+---------+---------------+---------------+-------------+-------------+



In [15]:
from pyspark.sql import functions as F
# 1. Ordenar por valor de forma descendente
# 2. Tomar el primer registro
ganador = df.orderBy(F.desc("valor")).limit(1)

ganador.select("item", "valor", "grupo", "fecha_captura").show()

+--------------------+-----+------------+-------------------+
|                item|valor|       grupo|      fecha_captura|
+--------------------+-----+------------+-------------------+
|Full Moon over N ...|49.43|G1_Ecommerce|2026-04-08 13:28:27|
+--------------------+-----+------------+-------------------+



In [16]:
from pyspark.sql import functions as F

NOMBRE_GRUPO = "Real Estate"

# CONSULTA DE SALIDA: Resumen de integridad por Categoría
ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()

--- TICKET DE SALIDA: Real Estate ---
+---------+------------+------------+---------------------+
|categoria|total_libros|precio_medio|ultima_sincronizacion|
+---------+------------+------------+---------------------+
+---------+------------+------------+---------------------+

